# 21.4 Description logics and ontologies

Ontologies trade some expressiveness for reliable classification: concepts, roles, subsumption, and consistency checks.

Description logics make the class-and-role part of first-order logic precise enough for ontology reasoners. Concepts are sets, roles are pairs, and axioms drive classification and validation.

Save a copy to Drive to edit.

In [ ]:

import random

import matplotlib.pyplot as plt

random.seed(2104)


def closure_subclasses(subclass):
    closure = {child: set(parents) for child, parents in subclass.items()}
    changed = True
    while changed:
        changed = False
        for child, parents in list(closure.items()):
            extra = set()
            for parent in parents:
                extra |= closure.get(parent, set())
            if not extra.issubset(parents):
                closure[child] |= extra
                changed = True
    return closure


def classify_ontology(concepts, roles, axioms):
    inferred = {name: set(values) for name, values in concepts.items()}
    checks = 0
    violations = []
    closure = closure_subclasses(axioms.get("subclass", {}))
    for child, parents in closure.items():
        child_values = inferred.get(child, set())
        for parent in parents:
            checks += 1
            inferred.setdefault(parent, set()).update(child_values)
    for left, right in axioms.get("disjoint", []):
        checks += 1
        overlap = inferred.get(left, set()) & inferred.get(right, set())
        if overlap:
            violations.append((left, right, overlap))
    existential = {}
    for name, role_name, filler in axioms.get("exists", []):
        checks += 1
        filler_values = inferred.get(filler, set())
        role_pairs = roles.get(role_name, set())
        existential[name] = {subject for subject, obj in role_pairs if obj in filler_values}
        inferred[name] = set(existential[name])
    return {"concepts": inferred, "violations": violations, "existential": existential, "checks": checks, "satisfiable": len(violations) == 0}


def make_dl_ladder():
    return [
        {"name": "D1 professor person", "concepts": {"Professor": {"Ada", "Cy"}, "Person": {"Ada", "Bo", "Cy"}, "Student": {"Bo", "Cy"}, "Employee": {"Ada", "Cy"}, "Cat": set(), "Dog": set()}, "roles": {"hasChild": {("Ann", "Cal"), ("Bob", "Dan")}}, "axioms": {"subclass": {"Professor": {"Person"}}, "disjoint": [("Cat", "Dog")], "exists": [("ParentOfDoctor", "hasChild", "Doctor")]}, "expected": True},
        {"name": "D2 existential role", "concepts": {"Doctor": {"Cal"}, "Professor": {"Ada", "Cy"}, "Person": {"Ada", "Bo", "Cy", "Cal"}}, "roles": {"hasChild": {("Ann", "Cal"), ("Bob", "Dan")}}, "axioms": {"subclass": {"Professor": {"Person"}}, "disjoint": [], "exists": [("ParentOfDoctor", "hasChild", "Doctor")]}, "expected": True},
        {"name": "D3 disjoint conflict", "concepts": {"Cat": {"Milo"}, "Dog": {"Milo"}, "Animal": {"Milo"}}, "roles": {}, "axioms": {"subclass": {"Cat": {"Animal"}, "Dog": {"Animal"}}, "disjoint": [("Cat", "Dog")], "exists": []}, "expected": False},
        {"name": "D4 university ontology", "concepts": {"Professor": {"Ada", "Cy"}, "Student": {"Bo", "Di"}, "Employee": {"Ada", "Cy", "Eli"}, "Person": {"Ada", "Bo", "Cy", "Di", "Eli"}}, "roles": {"advises": {("Ada", "Bo"), ("Cy", "Di")}}, "axioms": {"subclass": {"Professor": {"Employee", "Person"}, "Student": {"Person"}}, "disjoint": [], "exists": [("Advisor", "advises", "Student")]}, "expected": True},
        {"name": "D5 product medical ontology", "concepts": {"CardiologyDevice": {"D1", "D2"}, "Implant": {"D2"}, "Recalled": {"D2"}, "Product": {"D1", "D2", "D3"}, "Patient": {"P1", "P2"}, "Doctor": {"Cal", "Dana"}}, "roles": {"prescribedBy": {("P1", "Cal"), ("P2", "Dana")}, "usesDevice": {("P1", "D1"), ("P2", "D2")}}, "axioms": {"subclass": {"CardiologyDevice": {"MedicalDevice"}, "MedicalDevice": {"Product"}, "Implant": {"MedicalDevice"}}, "disjoint": [("Recalled", "Approved")], "exists": [("DoctorPatient", "prescribedBy", "Doctor"), ("DeviceUser", "usesDevice", "MedicalDevice")]}, "expected": True},
    ]


## The concept, built once (D1)

A concept $C$ denotes a set $C^I$. Subsumption $Professor\sqsubseteq Person$ means $Professor^I\subseteq Person^I$, while disjointness requires intersections such as $Cat\sqcap Dog$ to have size $0$.

In [ ]:

concepts = {"Professor": {"Ada", "Cy"}, "Person": {"Ada", "Bo", "Cy"}, "Student": {"Bo", "Cy"}, "Employee": {"Ada", "Cy"}, "Cat": {"Milo"}, "Dog": {"Milo"}, "Doctor": {"Cal"}}
roles = {"hasChild": {("Ann", "Cal"), ("Bob", "Dan")}}
axioms = {"subclass": {"Professor": {"Person"}}, "disjoint": [("Cat", "Dog")], "exists": [("ParentOfDoctor", "hasChild", "Doctor")]}

d1 = classify_ontology(concepts, roles, axioms)
professor_subset = concepts["Professor"].issubset(concepts["Person"])
student_employee = concepts["Student"] & concepts["Employee"]

print("professor subset", professor_subset)
print("student employee", student_employee)
print("violations", d1["violations"])
print("parent of doctor", d1["existential"]["ParentOfDoctor"])


The lesson numbers are exact: $2$ professors are contained in $3$ people, $Student\sqcap Employee$ has size $1$, $Cat\sqcap Dog=\{Milo\}$ is $1$ violation instead of required size $0$, and $\exists hasChild.Doctor=\{Ann\}$ has size $1$.

In [ ]:

assert professor_subset is True
assert len(concepts["Professor"]) == 2
assert len(concepts["Person"]) == 3
assert len(student_employee) == 1
assert len(d1["violations"]) == 1
assert len(d1["existential"]["ParentOfDoctor"]) == 1
assert "Ann" in d1["existential"]["ParentOfDoctor"]


## The dataset ladder

D1-D5 grow from tiny set containment to deep subclassing, existential roles, and consistency checks over product and medical concepts.

In [ ]:

dl_ladder = make_dl_ladder()

for rung in dl_ladder:
    concept_count = len(rung["concepts"])
    individual_count = len(set().union(*rung["concepts"].values()))
    print(rung["name"], "concepts", concept_count, "individuals", individual_count, "axioms", rung["axioms"])


## Run the same method across D1-D5

Each rung classifies subclass closures, evaluates existential roles, and checks disjointness. The metric combines satisfiability, expected correctness, and axiom checks.

In [ ]:

dl_results = []
for rung in dl_ladder:
    result = classify_ontology(rung["concepts"], rung["roles"], rung["axioms"])
    correct = result["satisfiable"] == rung["expected"]
    dl_results.append({
        "name": rung["name"],
        "size": len(rung["concepts"]),
        "checks": result["checks"],
        "violations": len(result["violations"]),
        "correct": correct,
        "result": result,
    })

for row in dl_results:
    print(row["name"], row["size"], row["checks"], row["violations"], row["correct"])


## Results visualization

The panels summarize hierarchy and inconsistency evidence. The curve shows checks and violations as ontology size grows.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()

for ax, row in zip(flat_axes[:5], dl_results):
    text = f"satisfiable={row['result']['satisfiable']}\nviolations={row['result']['violations']}\nexists={row['result']['existential']}"
    ax.text(0.03, 0.95, text, va="top", fontsize=8)
    ax.set_title(row["name"])
    ax.axis("off")

ax = flat_axes[5]
ax.plot([row["size"] for row in dl_results], [row["checks"] for row in dl_results], marker="o", label="checks")
ax.plot([row["size"] for row in dl_results], [row["violations"] for row in dl_results], marker="x", label="violations")
ax.set_xlabel("concept count")
ax.set_title("checks and violations")
ax.legend()
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: confusing a surviving candidate with a proved answer. D5 requires every subsumption, role, and disjointness axiom to be checked before accepting a classification.

In [ ]:

hard = dl_ladder[-1]
naive_survivor = "P2" in hard["concepts"]["Patient"]
fixed = classify_ontology(hard["concepts"], hard["roles"], hard["axioms"])

print("naive survivor", naive_survivor)
print("checked axioms", fixed["checks"])
print("device users", fixed["existential"].get("DeviceUser"))


## Evaluate it + Practice

- Metric: satisfiability, classification correctness, and axiom-check count, compared with a no-skill baseline that guesses or accepts the first candidate.
- Sanity check: rerun D1 and verify the hand-counted lesson numbers before trusting larger rungs.
- Ablation: turn off the key symbolic check and confirm the metric drops on D3-D5.
- Failure signals: unexplained answers, fewer checked cases than the rung size requires, or a contradiction accepted as valid.
- Keep all instances CPU-only, seeded, and small enough to inspect.

Practice: Add Approved={D2} to D5 and predict the disjointness result.

Practice: Add a two-step subclass chain and verify inherited Product membership.

Practice: Create an existential role query that returns the empty set and explain why.